# LSYNTH process metric between two datasets using standalone LSM

This notebook computes the LSYNTH dataset/process metric while replacing the historical QuasiNet conditional learner with the current standalone C++ LSM implementation from `zeroknowledgediscovery/lsm` (`dev_ixc`).

For a fitted conditional model $G$ and state $x$, LSYNTH defines the normalized conditional profile

$$
u_G(x,i)
=
\frac{\phi_G^i(x_i\mid x_{-i})}
{\max_y \phi_G^i(y\mid x_{-i})}.
$$

Given two datasets $D_1,D_2$, fit one LSM to each dataset on the same feature set. On a common held-out reference sample $E_\mu$, estimate

$$
\boxed{
\widehat d_\mu(D_1,D_2)
=
\frac{1}{M}
\sum_{x\in E_\mu}
\frac{1}{|I_x|}
\sum_{i\in I_x}
\left|u_{G_1}(x,i)-u_{G_2}(x,i)\right|
}
$$

where $I_x$ contains coordinates observed in the reference row and evaluable by both models. Thus $0\le \widehat d_\mu\le1$, with smaller values indicating more similar learned conditional structure.

To keep evaluator fitting separate from evaluation, each dataset is split into fit and holdout partitions. The practical reference measure is a balanced held-out mixture,

$$
\mu \approx \tfrac12 P_1^{\rm holdout}+\tfrac12 P_2^{\rm holdout}.
$$

This is an empirical approximation to the fixed-reference LSYNTH metric. The strict population metric theorem additionally assumes a full-support reference measure and strictly positive conditional kernels.

In [ ]:
from pathlib import Path
import importlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LSYNTH_ROOT = Path('/home/ishanu/Dropbox/ZED/Research/lsynth')
LSM_ROOT = Path('/home/ishanu/Dropbox/ZED/Research/lsm')

sys.path.insert(0, str(LSYNTH_ROOT))
sys.path.insert(0, str(LSM_ROOT / 'bin'))

import lsynth.lsm_metric as lm
importlib.reload(lm)

print('LSYNTH helper:', lm.__file__)
print('LSM trainer:', LSM_ROOT / 'bin' / 'LSM')


## Configuration

Change only the two dataset paths to compare another pair. The models are trained on the shared columns in the same order.

In [ ]:
GSS_DIR = Path('/home/ishanu/Dropbox/ZED/Research/MAGICS_research/survey/data/gss')

# Example pair; replace with any two categorical CSV datasets.
DATASET_A = GSS_DIR / 'gss_1980.csv'
DATASET_B = GSS_DIR / 'gss_2018.csv'
LABEL_A = DATASET_A.stem
LABEL_B = DATASET_B.stem

FIT_FRACTION = 0.50
REFERENCE_ROWS_PER_DATASET = 100
FEATURE_LIMIT = 0       # 0 = all shared columns
SEED = 20260922

# Standalone LSM training
LSM_ALPHA = 0.10
LSM_THREADS = 12
SUBSET_MODE = 'auto'
MAX_EXACT_LEVELS = 20
FAST_LEVELS = 16
OVERWRITE_MODELS = False

# Metric evaluation
PROFILE_WORKERS = 1     # safest default; increase after a successful smoke run
EPS_FLOOR = 1e-12       # tiny positive floor for target labels absent from one model
BOOTSTRAPS = 2000
CI = 0.95

OUTDIR = LSYNTH_ROOT / 'results' / f'lsm_metric_{LABEL_A}_vs_{LABEL_B}'
OUTDIR.mkdir(parents=True, exist_ok=True)

print('A:', DATASET_A)
print('B:', DATASET_B)
print('output:', OUTDIR)


## Load datasets and define the common state space

LSYNTH compares conditional systems on a common product space, so both LSMs are trained on exactly the same shared columns. Dataset A's column order is used.

In [ ]:
df_a_raw = lm.read_categorical_csv(DATASET_A)
df_b_raw = lm.read_categorical_csv(DATASET_B)

features = lm.common_features(
    df_a_raw,
    df_b_raw,
    feature_limit=FEATURE_LIMIT,
)

df_a = df_a_raw.loc[:, features].copy()
df_b = df_b_raw.loc[:, features].copy()

print(f'{LABEL_A}: {df_a.shape}')
print(f'{LABEL_B}: {df_b.shape}')
print('shared features:', len(features))
display(pd.DataFrame({'feature': features}).head(20))


## Split model-fitting rows from the common reference sample

Each LSM is fit only on its own fit partition. The reference sample contains equal numbers of held-out rows from A and B and is not used to train either evaluator.

In [ ]:
fit_a, holdout_a = lm.split_fit_holdout(
    df_a, fit_fraction=FIT_FRACTION, seed=SEED + 1
)
fit_b, holdout_b = lm.split_fit_holdout(
    df_b, fit_fraction=FIT_FRACTION, seed=SEED + 2
)

reference_with_source = lm.balanced_reference_sample(
    holdout_a,
    holdout_b,
    rows_per_dataset=REFERENCE_ROWS_PER_DATASET,
    seed=SEED + 3,
)
reference_source = reference_with_source['__reference_source__'].copy()
reference = reference_with_source.drop(columns='__reference_source__')

print('fit A:', fit_a.shape, 'holdout A:', holdout_a.shape)
print('fit B:', fit_b.shape, 'holdout B:', holdout_b.shape)
print('reference:', reference.shape)
print(reference_source.value_counts().to_dict())


## Train or reuse the two standalone LSMs

The standard `dev_ixc` trainer is used. Each model contains binary trees plus source maps and is therefore inference-ready for raw string rows.

In [ ]:
model_a = OUTDIR / 'model_A'
model_b = OUTDIR / 'model_B'

lm.train_lsm(
    fit_a,
    lsm_root=LSM_ROOT,
    model_dir=model_a,
    train_csv=OUTDIR / 'fit_A.csv',
    alpha=LSM_ALPHA,
    threads=LSM_THREADS,
    subset_mode=SUBSET_MODE,
    max_exact_levels=MAX_EXACT_LEVELS,
    fast_levels=FAST_LEVELS,
    overwrite=OVERWRITE_MODELS,
)

lm.train_lsm(
    fit_b,
    lsm_root=LSM_ROOT,
    model_dir=model_b,
    train_csv=OUTDIR / 'fit_B.csv',
    alpha=LSM_ALPHA,
    threads=LSM_THREADS,
    subset_mode=SUBSET_MODE,
    max_exact_levels=MAX_EXACT_LEVELS,
    fast_levels=FAST_LEVELS,
    overwrite=OVERWRITE_MODELS,
)

print('model A:', model_a)
print('model B:', model_b)


## Evaluate the two normalized conditional profiles on the same $E_\mu$

For every held-out reference row and every observed target coordinate, the standalone LSM returns $\phi^i(\cdot\mid x^{-i})$. The observed-label probability is divided by the modal probability to obtain the LSYNTH $u_G(x,i)$ score.

In [ ]:
profile_a = lm.evaluate_lsm_profiles(
    reference,
    model_dir=model_a,
    lsm_root=LSM_ROOT,
    profile_workers=PROFILE_WORKERS,
    eps_floor=EPS_FLOOR,
    progress=True,
)

profile_b = lm.evaluate_lsm_profiles(
    reference,
    model_dir=model_b,
    lsm_root=LSM_ROOT,
    profile_workers=PROFILE_WORKERS,
    eps_floor=EPS_FLOOR,
    progress=True,
)

print('profile A:', profile_a.shape)
print('profile B:', profile_b.shape)


## Compute $\widehat d_\mu$ and a reference-row bootstrap interval

The bootstrap resamples complete held-out reference rows. This matches the LSYNTH estimator's sampling unit.

In [ ]:
row_distance = lm.row_profile_distances(profile_a, profile_b)
valid = np.isfinite(row_distance)

boot = lm.bootstrap_metric(
    row_distance,
    n_bootstrap=BOOTSTRAPS,
    ci=CI,
    seed=SEED + 4,
)

summary = pd.DataFrame([{
    'dataset_a': LABEL_A,
    'dataset_b': LABEL_B,
    'd_mu_hat': boot.estimate,
    'ci_low': boot.ci_low,
    'ci_high': boot.ci_high,
    'n_reference_rows': len(reference),
    'n_valid_reference_rows': int(valid.sum()),
    'n_features': len(features),
}])

display(summary)
print(
    f"LSYNTH-LSM d_mu({LABEL_A}, {LABEL_B}) = "
    f"{boot.estimate:.6f}  "
    f"95% CI [{boot.ci_low:.6f}, {boot.ci_high:.6f}]"
)


## Which coordinates contribute most?

The LSYNTH distance is an average absolute difference of normalized conditional profiles, so its coordinate contributions are directly decomposable.

In [ ]:
contrib = lm.coordinate_contributions(profile_a, profile_b)
contrib_df = pd.DataFrame({
    'feature': features,
    'mean_abs_profile_difference': contrib,
}).sort_values('mean_abs_profile_difference', ascending=False, na_position='last')

display(contrib_df.head(25))


## Row-level distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.hist(row_distance[valid], bins=25)
ax.axvline(boot.estimate, linestyle='--', linewidth=1.2)
ax.set_xlabel(r'Row profile distance $N_x^{-1}\sum_i |u_A-u_B|$')
ax.set_ylabel('Reference rows')
ax.set_title(f'LSYNTH-LSM process distance: {LABEL_A} vs {LABEL_B}')
fig.tight_layout()
plt.show()


## Save results

In [ ]:
summary.to_csv(OUTDIR / 'lsynth_lsm_metric_summary.csv', index=False)
contrib_df.to_csv(OUTDIR / 'lsynth_lsm_metric_coordinate_contributions.csv', index=False)

row_df = pd.DataFrame({
    'reference_source': reference_source,
    'row_profile_distance': row_distance,
})
row_df.to_csv(OUTDIR / 'lsynth_lsm_metric_row_distances.csv', index=False)
reference_with_source.to_csv(OUTDIR / 'reference_rows.csv', index=False)

metadata = {
    'dataset_a': str(DATASET_A),
    'dataset_b': str(DATASET_B),
    'fit_fraction': FIT_FRACTION,
    'reference_rows_per_dataset': REFERENCE_ROWS_PER_DATASET,
    'n_features': len(features),
    'features': features,
    'lsm_alpha': LSM_ALPHA,
    'lsm_threads': LSM_THREADS,
    'subset_mode': SUBSET_MODE,
    'max_exact_levels': MAX_EXACT_LEVELS,
    'fast_levels': FAST_LEVELS,
    'eps_floor': EPS_FLOOR,
    'bootstrap_replicates': BOOTSTRAPS,
    'seed': SEED,
}
(OUTDIR / 'metadata.json').write_text(json.dumps(metadata, indent=2))

print(OUTDIR / 'lsynth_lsm_metric_summary.csv')
print(OUTDIR / 'lsynth_lsm_metric_coordinate_contributions.csv')
print(OUTDIR / 'lsynth_lsm_metric_row_distances.csv')
print(OUTDIR / 'reference_rows.csv')
print(OUTDIR / 'metadata.json')
